In [ ]:
# import collections, re, json
# from openai import OpenAI

# client = OpenAI()
# Q = "列出 Fisher 信息有限的条件下 Cramér-Rao 不等式成立所需的全部假设。"
# votes = collections.Counter()
# for _ in range(8):
#     r = client.chat.completions.create(
#         model="gpt-4o", messages=[{"role": "user", "content": Q}], temperature=0.8)
#     # 抽取每条假设的短标签，结构化为可比较的项
#     for line in r.choices[0].message.content.splitlines():
#         tag = re.sub(r"^[0-9.\-•\s]+", "", line).strip()[:24]
#         if tag:
#             votes[tag] += 1
# print([k for k, v in votes.items() if v >= 5])   # 高票项进入假设清单
# print([k for k, v in votes.items() if v <= 2])   # 低票项交由人工核对原文

In [2]:
import numpy as np

rng = np.random.default_rng(20260925)

# 一条固定路径：X_i ~ N(0,1)，看累积样本均值 X̄_n
n_max, path_len = 4000, 4000
X = rng.standard_normal(path_len)
xbar = np.cumsum(X) / np.arange(1, path_len + 1)

print("命题 A 的 N(ε)：首个使 |X̄_m| < ε 对一切 m∈[N, N+200] 成立的 N")
print(f"{'ε':>8}{'N(ε)':>10}{'N(ε) 处的 |X̄|':>16}")
for eps in (0.5, 0.2, 0.1, 0.05, 0.02):
    ok = np.abs(xbar) < eps
    # 滑窗检查：从位置 N 起连续 200 项都满足
    N_found = None
    for N in range(0, path_len - 200):
        if ok[N:N + 200].all():
            N_found = N
            break
    if N_found is None:
        print(f"{eps:>8}{'未找到':>10}")
    else:
        print(f"{eps:>8}{N_found:>10}{abs(xbar[N_found]):>16.4f}")

# 两变量版本：∀x ∃y (y² ≥ x) 与 ∃y ∀x (y² ≥ x)，x 取遍非负实数
xs = np.linspace(0.0, 100.0, 1001)
print("\n两变量版本：固定网格 [0, 100]，比较两种量词顺序")
print("∀x ∃y (y²≥x)：成立（对每个 x 取 y=√x）")
for y in (1.0, 5.0, 10.0, 50.0):
    print(f"  固定 y={y:>5}: 网格覆盖率 = {np.mean(y**2 >= xs):.3f}，"
          f"失效于任何 x > {y**2:.0f}")
print("∃y ∀x (y²≥x)：在无界定义域上不成立，任何固定 y 都被 x = y²+1 击穿")

命题 A 的 N(ε)：首个使 |X̄_m| < ε 对一切 m∈[N, N+200] 成立的 N
       ε      N(ε)    N(ε) 处的 |X̄|
     0.5         8          0.4130
     0.2        14          0.1823
     0.1       363          0.0951
    0.05       443          0.0475
    0.02      1317          0.0198

两变量版本：固定网格 [0, 100]，比较两种量词顺序
∀x ∃y (y²≥x)：成立（对每个 x 取 y=√x）
  固定 y=  1.0: 网格覆盖率 = 0.011，失效于任何 x > 1
  固定 y=  5.0: 网格覆盖率 = 0.251，失效于任何 x > 25
  固定 y= 10.0: 网格覆盖率 = 1.000，失效于任何 x > 100
  固定 y= 50.0: 网格覆盖率 = 1.000，失效于任何 x > 2500
∃y ∀x (y²≥x)：在无界定义域上不成立，任何固定 y 都被 x = y²+1 击穿


In [3]:
import numpy as np

rng = np.random.default_rng(20260924)
n_reps = 200_000

for dist_name, samples in [
    ("N(0,1)",      rng.normal(0, 1, n_reps)),
    ("Exp(1)",      rng.exponential(1, n_reps) - 1.0),
    ("两点分布",     rng.choice([-1.0, 1.0], size=n_reps)),
]:
    m = samples.mean()
    var = samples.var()
    for eps in (0.5, 1.0, 2.0):
        emp = np.mean(np.abs(samples - m) >= eps)
        bound = min(1.0, var / eps**2)          # Chebyshev：Var / eps^2
        flag = "越界" if emp > bound + 1e-3 else ""
        print(f"{dist_name:8s} eps={eps:>4}  经验={emp:.5f}  Chebyshev界={bound:.5f}  {flag}")
# 用途：若出现"经验 > 界"，先怀疑陈述写反或常数写错，再怀疑分布实现

N(0,1)   eps= 0.5  经验=0.61759  Chebyshev界=1.00000  
N(0,1)   eps= 1.0  经验=0.31833  Chebyshev界=1.00000  
N(0,1)   eps= 2.0  经验=0.04512  Chebyshev界=0.25038  
Exp(1)   eps= 0.5  经验=0.61682  Chebyshev界=1.00000  
Exp(1)   eps= 1.0  经验=0.13658  Chebyshev界=1.00000  
Exp(1)   eps= 2.0  经验=0.04953  Chebyshev界=0.25029  
两点分布     eps= 0.5  经验=1.00000  Chebyshev界=1.00000  
两点分布     eps= 1.0  经验=0.49933  Chebyshev界=1.00000  
两点分布     eps= 2.0  经验=0.00000  Chebyshev界=0.25000  


In [4]:
import sympy as sp

n = sp.symbols('n', positive=True, integer=True)
sigma2 = sp.symbols('sigma^2', positive=True)

# 1) 有偏版本的偏差：E[S_n^2] - sigma^2
bias = sp.simplify(((n - 1) / n) * sigma2 - sigma2)
print("偏差 E[S_n^2] - sigma^2 =", sp.factor(bias))      # 期望输出 -sigma^2/n
print("n -> oo 时偏差的极限 =", sp.limit(bias, n, sp.oo))  # 期望输出 0（渐近无偏）

# 2) 无偏版本的检查：乘 n/(n-1) 之后是否回到 sigma^2
print("乘 n/(n-1) 后 =", sp.simplify(((n - 1) / n) * sigma2 * n / (n - 1)))

# 3) 对具体分布做符号验证：X ~ Bernoulli(p)，n = 3
p = sp.symbols('p', positive=True)
X = sp.symbols('X0 X1 X2')
vals = [(x, p, 1) for x in X[:1]]  # 占位：见下文的直接枚举
# 直接枚举 2^3 = 8 个样本点，加权求和（避免依赖 SymPy 的求和化简能力）
import itertools
xs = [sp.Integer(0), sp.Integer(1)]
E_S2 = 0
for combo in itertools.product(xs, repeat=3):
    w = sp.prod([p if c == 1 else (1 - p) for c in combo])
    xbar = sp.Rational(1, 3) * sum(combo)
    S2 = sp.Rational(1, 3) * sum((sp.Integer(c) - xbar) ** 2 for c in combo)
    E_S2 += sp.expand(w * S2)
print("Bernoulli(p), n=3: E[S_3^2] =", sp.factor(E_S2), " sigma^2 =", sp.factor(p * (1 - p)))
print("比值 =", sp.simplify(E_S2 / (p * (1 - p))))   # 期望输出 2/3 = (n-1)/n

偏差 E[S_n^2] - sigma^2 = -sigma^2/n
n -> oo 时偏差的极限 = 0
乘 n/(n-1) 后 = sigma^2
Bernoulli(p), n=3: E[S_3^2] = -2*p*(p - 1)/3  sigma^2 = -p*(p - 1)
比值 = 2/3


In [5]:
# 层一：单样本 t 检验在原假设下的拒绝概率（数值直觉检查）
import numpy as np
from scipy import stats

rng = np.random.default_rng(20260924)
alpha, mu0, sigma, n, reps = 0.05, 0.0, 1.0, 10, 200_000

# 每次重复生成一个样本并计算 t 统计量（用向量化加速）
X = rng.normal(mu0, sigma, size=(reps, n))
xbar = X.mean(axis=1)
s2 = X.var(axis=1, ddof=1)                    # ddof=1 即除以 n-1
t = (xbar - mu0) / np.sqrt(s2 / n)
crit = stats.t.ppf(1 - alpha / 2, df=n - 1)   # 双边检验的临界值
reject_rate = np.mean(np.abs(t) > crit)
print(f"n={n}, alpha={alpha}: 经验拒绝概率 = {reject_rate:.4f} "
      f"(蒙特卡洛标准误 ≈ {np.sqrt(reject_rate*(1-reject_rate)/reps):.4f})")

# 边界抽查：若把 ddof 写成 0（除以 n 的错误版本），拒绝概率会偏离 alpha
t_wrong = (xbar - mu0) / np.sqrt(X.var(axis=1, ddof=0) / n)
print("错误版本(ddof=0)的拒绝概率 =", np.mean(np.abs(t_wrong) > crit).round(4))
# 用途：这一步不能证明任何事，但能立刻发现自由度、单双边、ddof 之类的低级错误

n=10, alpha=0.05: 经验拒绝概率 = 0.0492 (蒙特卡洛标准误 ≈ 0.0005)
错误版本(ddof=0)的拒绝概率 = 0.0594


In [6]:
# Hoeffding 界的数值检查：经验尾部概率 vs 界
import numpy as np

rng = np.random.default_rng(7)
reps, n = 100_000, 20
# X_i 独立，X_i ∈ [a_i, b_i]，取 a=0, b=1 的 Bernoulli(0.5)
a, b, p = 0.0, 1.0, 0.5
X = (rng.random((reps, n)) < p).astype(float)
S = X.sum(axis=1)
ES = n * p
denom = n * (b - a) ** 2

print(f"{'t':>6} {'经验 P(S-ES>=t)':>18} {'Hoeffding 界':>14} {'是否越界':>10}")
for t in (1.0, 2.0, 3.0, 4.0, 5.0):
    emp = np.mean(S - ES >= t)
    bound = np.exp(-2 * t**2 / denom)
    print(f"{t:>6.1f} {emp:>18.5f} {bound:>14.5f} {'是' if emp > bound + 1e-9 else '否'}")
# 用途：出现"越界"说明陈述（而非证明）写错了，例如常数位置、单双边、区间长度

     t      经验 P(S-ES>=t)    Hoeffding 界       是否越界
   1.0            0.41091        0.90484 否
   2.0            0.25173        0.67032 否
   3.0            0.13254        0.40657 否
   4.0            0.05874        0.20190 否
   5.0            0.02076        0.08208 否


In [7]:
import numpy as np

rng = np.random.default_rng(20260925)
reps = 5000                      # 每个重复生成一个 n 样本并算样本均值
n_list = [10, 100, 1000, 10000]

print(f"{'分布':<22}{'n':>7}{'均值':>10}{'样本间sd':>11}{'P(|X̄-μ|>0.1)':>14}{'Chebyshev界':>12}")
print("-" * 78)

# 三个满足 E|X| < ∞ 的分布（前两个方差有限，第三个方差无穷）
cases = [
    ("N(0,1) 方差有限",       lambda s: rng.standard_normal(s),            0.0, 1.0),
    ("Exp(1)-1 方差有限",     lambda s: rng.exponential(1.0, s) - 1.0,     0.0, 1.0),
    ("Pareto(1.5)-3 均值有限", lambda s: (1.0 / rng.uniform(0, 1, s)) ** (1 / 1.5) - 3.0, 0.0, np.inf),
]
for name, gen, mu, var in cases:
    for n in n_list:
        m = gen((reps, n)).mean(axis=1)
        tail = np.mean(np.abs(m - mu) > 0.1)
        cheb = min(1.0, var / (n * 0.1**2)) if np.isfinite(var) else float('nan')
        print(f"{name:<22}{n:>7}{m.mean():>10.4f}{m.std():>11.4f}{tail:>14.4f}"
              f"{('%.4f' % cheb) if np.isfinite(cheb) else '      —':>12}")
    print()

# Cauchy(0,1)：E|X| = ∞，WLLN 前件不满足；样本均值对每个 n 仍精确服从 Cauchy
print("Cauchy(0,1)  E|X|=∞（WLLN 前件不满足）")
for n in n_list:
    m = rng.standard_cauchy((reps, n)).mean(axis=1)
    print(f"{'':<22}{n:>7}  中位数={np.median(m):>8.4f}  "
          f"P(|X̄|>1)={np.mean(np.abs(m) > 1):.4f}  P(|X̄|>10)={np.mean(np.abs(m) > 10):.4f}")

分布                          n        均值      样本间sd P(|X̄-μ|>0.1)  Chebyshev界
------------------------------------------------------------------------------
N(0,1) 方差有限                10   -0.0050     0.3102        0.7526      1.0000
N(0,1) 方差有限               100   -0.0002     0.0999        0.3086      1.0000
N(0,1) 方差有限              1000    0.0009     0.0312        0.0016      0.1000
N(0,1) 方差有限             10000    0.0002     0.0100        0.0000      0.0100

Exp(1)-1 方差有限              10   -0.0003     0.3209        0.7612      1.0000
Exp(1)-1 方差有限             100    0.0020     0.1029        0.3194      1.0000
Exp(1)-1 方差有限            1000   -0.0001     0.0318        0.0030      0.1000
Exp(1)-1 方差有限           10000   -0.0001     0.0099        0.0000      0.0100

Pareto(1.5)-3 均值有限         10    0.2326    19.3023        0.9546           —
Pareto(1.5)-3 均值有限        100   -0.0652     1.1335        0.9062           —
Pareto(1.5)-3 均值有限       1000   -0.0120     0.7456        0.7896        

In [ ]:
# import json, subprocess, pathlib, hashlib, datetime
# from openai import OpenAI

# client = OpenAI()
# LEAN_ENV = ["lake", "env", "lean", "--stdin"]     # 以项目根目录为 cwd
# TMPL = """你是一名形式化助手。请把下面的自然语言命题翻译为 Lean 4 陈述。
# 要求：
# 1. 只输出 theorem/lemma 的陈述与 `by sorry`，不要写证明；
# 2. 把所有隐含条件写成显式参数；
# 3. 若某条前件不确定是否需要，就写上，并在注释里说明。
# 命题：{nl}
# """

# def formalize_once(nl: str, model: str, seed: int) -> str:
#     r = client.chat.completions.create(
#         model=model, seed=seed, temperature=0.0,
#         messages=[{"role": "user", "content": TMPL.format(nl=nl)}])
#     return r.choices[0].message.content

# def compiles(src: str, cwd: str) -> tuple[bool, str]:
#     """把单条陈述包进最小上下文后编译；返回 (是否通过, 报错原文)"""
#     p = subprocess.run(LEAN_ENV, input="import Mathlib\n" + src,
#                        cwd=cwd, capture_output=True, text=True, timeout=300)
#     return (p.returncode == 0 and "sorry" in src), (p.stdout + p.stderr)[-2000:]

# def signature_of(src: str) -> str:
#     """极粗糙的签名提取：去掉空白与注释后取前 200 字符，用于两条陈述的快速比对"""
#     s = "".join(src.split())
#     return hashlib.md5(s.encode()).hexdigest()[:8] + ":" + s[:200]

# propositions = json.load(open("propositions.json"))   # [{"id":..., "nl":...}, ...]
# records = []
# for prop in propositions:
#     a = formalize_once(prop["nl"], "model-A", seed=1)
#     b = formalize_once(prop["nl"], "model-B", seed=2)
#     oka, erra = compiles(a, cwd="stat_check")
#     okb, errb = compiles(b, cwd="stat_check")
#     records.append({
#         "id": prop["id"], "nl": prop["nl"],
#         "A": {"src": a, "compiles": oka, "err": erra, "sig": signature_of(a)},
#         "B": {"src": b, "compiles": okb, "err": errb, "sig": signature_of(b)},
#         "agree_signature": signature_of(a) == signature_of(b),
#         "reviewed_by": None,            # 人工复核签字位，为空则不归档
#         "timestamp": datetime.date.today().isoformat(),
#     })

# pathlib.Path("logs").mkdir(exist_ok=True)
# json.dump(records, open(f"logs/dual-{datetime.date.today()}.json", "w"),
#           ensure_ascii=False, indent=2)
# # 说明：agree_signature == True 只说明两条陈述字面接近，不等于忠实；
# #       所有记录都须经人工复核（reviewed_by 非空）才进入归档。

In [9]:
import numpy as np
import sympy as sp

# 例：检验"对任意分布，样本方差的无偏修正因子为 (n-1)/n"这一说法在 n=1 处的边界行为
n, s = sp.symbols('n s', positive=True)
# 猜想形式化为 R(n) = E[S_n^2] - sigma^2，若某 n 使 R != 0 则猜想的"无偏"表述不成立
R = sp.simplify(((n - 1) / n) * s**2 - s**2)
print(sp.simplify(R))          # 输出 -s**2/n，说明除以 n 的样本方差有偏
print(sp.limit(R, n, sp.oo))   # 偏差随 n → ∞ 消失，与渐近无偏一致

# 数值搜索：在参数网格上寻找使 R > 0 的点
rng = np.random.default_rng(0)
def objective(theta):
    # 待检验的统计猜想：此处以"某不等式 LHS - RHS"为占位
    a, b = theta
    return a * (1 - b) - (a + b) / 2      # 替换为真实猜想的目标函数

best, best_val = None, -np.inf
for _ in range(200000):
    theta = rng.uniform(-5, 5, size=2)
    val = objective(theta)
    if val > best_val:
        best, best_val = theta, val
print("最优候选:", best, "目标值:", best_val, "是否构成反例:", best_val > 0)

-s**2/n
0
最优候选: [ 4.96059543 -4.98905726] 目标值: 29.723520955403735 是否构成反例: True


In [10]:
import sympy as sp

x, a, b = sp.symbols('x a b', real=True)

# 检验一条积分不等式是否在某些参数下失效
expr = sp.integrate(sp.exp(-a * x**2), (x, -sp.oo, sp.oo))   # sqrt(pi)/sqrt(a)，要求 a>0
print(sp.simplify(expr))

# 用符号方式检查"Jensen 方向"：凸函数 phi(x)=x^2 时 E[phi(X)] - phi(E[X]) = Var(X) >= 0
p = sp.symbols('p', positive=True)
v0, v1 = sp.symbols('v0 v1', real=True)
EX  = p * v0 + (1 - p) * v1
EX2 = p * v0**2 + (1 - p) * v1**2
gap = sp.simplify(EX2 - EX**2)
print(sp.factor(gap))   # 输出 p*(1-p)*(v0-v1)^2 >= 0，确认方向

Piecewise((sqrt(pi)/sqrt(a), Abs(arg(a)) <= pi/2), (Integral(exp(-a*x**2), (x, -oo, oo)), True))
-p*(p - 1)*(v0 - v1)**2


In [11]:
# 两段式复验：高精算术确认符号 → 符号计算尝试证明/否证
import mpmath as mp
import sympy as sp

def refine(theta, objective, digits=50):
    """在高精度下重算目标函数；返回 (值, 是否严格为正)"""
    mp.mp.dps = digits
    val = objective(*[mp.mpf(str(t)) for t in theta])
    return val, val > mp.mpf('1e-30')

# 例：待检验的不等式 (x+y)^2 <= 2(x^2+y^2)，令 R = 2(x^2+y^2) - (x+y)^2
def R(x, y):
    return 2 * (x**2 + y**2) - (x + y)**2

theta = (0.1, -0.3)
val, positive = refine(theta, R)
print("高精重算 R =", mp.nstr(val, 20), " 严格为正:", positive)

# 符号侧：该式恒等于 (x-y)^2 >= 0，故不存在反例
x, y = sp.symbols('x y', real=True)
print("符号化简 R =", sp.factor(sp.simplify(2*(x**2+y**2) - (x+y)**2)))
# 用途：高精算术的作用是排除浮点假阳性；符号化简的作用是排除整块参数区域。
#       两者都不构成证明，只决定"值不值得投入形式化成本"。

高精重算 R = 0.16  严格为正: True
符号化简 R = (x - y)**2


In [12]:
import sympy as sp

def is_trivial(conjecture_expr, known_theorems):
    """若猜想可由已知恒等式化简为 True，则为重述型平凡"""
    return sp.simplify(conjecture_expr) == 0

def numerical_check(f, rng, n=10000):
    """在随机实例上检查猜想是否成立，返回反例或 None"""
    import numpy as np
    for _ in range(n):
        theta = rng.uniform(-10, 10, size=2)
        if f(*theta) > 1e-9:        # LHS - RHS > 0 意味着反例
            return theta
    return None

x, y = sp.symbols('x y', real=True)
cand = (x + y)**2 - (x**2 + y**2)          # 候选猜想：LHS - RHS >= 0?
print("是否重述型平凡:", is_trivial(cand, []))
import numpy as np
print("数值反例:", numerical_check(lambda a, b: -(a + b)**2 + (a**2 + b**2),
                                   np.random.default_rng(1)))

是否重述型平凡: False
数值反例: [-7.11680775  8.97298894]


In [13]:
import numpy as np

def experiment(n, d, s, reps=200, rng=None):
    """返回某估计量在给定 (n, d, s) 下的平均误差；s 为稀疏度"""
    rng = rng or np.random.default_rng(0)
    errs = []
    for _ in range(reps):
        beta = np.zeros(d); beta[:s] = 1.0
        X = rng.normal(size=(n, d)) / np.sqrt(n)
        y = X @ beta + rng.normal(scale=0.1, size=n)
        # 以最小二乘为占位估计量，实际替换为研究中的估计量
        beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
        errs.append(np.linalg.norm(beta_hat - beta))
    return float(np.mean(errs))

rng = np.random.default_rng(42)
for ratio in [0.5, 1.0, 2.0, 4.0, 8.0]:
    n, d, s = 200, int(200 / ratio), 5
    print(f"n/d = {ratio:>4}: 平均误差 = {experiment(n, d, s, rng=rng):.4f}")
# 用途：观察误差在哪个 n/d 比值处开始恶化，该比值即形式化时应重点检查的假设

n/d =  0.5: 平均误差 = 2.1171
n/d =  1.0: 平均误差 = 115.5007
n/d =  2.0: 平均误差 = 1.4285
n/d =  4.0: 平均误差 = 0.8081
n/d =  8.0: 平均误差 = 0.5347
